In [45]:
import os
import glob
from datetime import datetime

import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from libpysal.weights import DistanceBand
from esda.getisord import G_Local
from esda.moran import Moran_Local
from statsmodels.stats.multitest import multipletests

In [46]:
# =========================
# USER INPUTS
# =========================

# Same root folder used in Notebook 02 (02_Calc_Grid_Coverage)
INPUT_FOLDER = r"/Users/ks/Desktop/Wu/Testing"

# Notebook 02 outputs we're consuming
ZONAL_STATS_DIR = os.path.join(INPUT_FOLDER, "02_Ecostress_Zonal_Stats")   # per-date .gpkg files
FINAL_AVERAGES_GPKG = os.path.join(INPUT_FOLDER, "04_Final_Phase_Averages", "Final_Phase_Averages.gpkg")

GRID_ID_FIELD = "GRID_ID"
PER_DATE_VALUE_FIELD = "lst_mean"                                  # column written by Notebook 02's zonal stats step
PHASE_CATEGORIES = ["Night", "Morning", "Afternoon", "Evening"]    # columns in Final_Phase_Averages.gpkg

distance_bands = [1000, 1500, 2000]   # meters
target_epsg = 3310                    # CA Albers 83
permutations = 999
save_individual_scene_csv = True

# If the underlying raster/zonal-stats step wrote NaN/NoData as 0, exclude those 0s
treat_zero_as_nodata = True
zero_nodata_tolerance = 0.0

# Gi* / Moran settings
gi_p_source_for_bins = "p_sim"     # options: "p_sim", "p_norm"
apply_fdr_to_gi_bins = True        # keep true
apply_fdr_to_moran = True          # keep true


def create_results_structure(root_folder):
    """Creates 05_Hotspot_Analysis_Results alongside Notebook 02's numbered output folders,
    with separate Per_Date / Phase_Average subfolders for raw results and PNGs."""
    base_path = os.path.join(root_folder, "05_Hotspot_Analysis_Results")
    raw_path = os.path.join(base_path, "Raw_Results")
    summary_path = os.path.join(base_path, "Summary_Tables")
    png_path = os.path.join(base_path, "PNG_Maps")

    for sub in ["Per_Date", "Phase_Average"]:
        os.makedirs(os.path.join(raw_path, sub), exist_ok=True)
        os.makedirs(os.path.join(png_path, sub), exist_ok=True)
    os.makedirs(summary_path, exist_ok=True)

    return base_path, raw_path, summary_path, png_path


# =========================
# COLOR SCHEMES
# =========================

gi_color_map = {
    3: "#D62F27",
    2: "#ED7551",
    1: "#FAB984",
    0: "#ffffbf",
    -1: "#C0CCBE",
    -2: "#849EBA",
    -3: "#4575B5"
}

moran_color_map = {
    "High-High": "#F0B8B1",
    "Low-Low": "#b7d9e8",
    "High-Low": "#E01B1B",
    "Low-High": "#1b53e0",
    "Not Significant": "#FFFFFF"
}

In [47]:
# =========================
# HELPERS
# =========================

def find_all_gpkgs(root_folder):
    """Recursively finds all .gpkg files under root_folder (e.g. mirrors the
    Morning/merged, Afternoon/merged, ... subfolders written by Notebook 02)."""
    gpkg_paths = []
    for dirpath, dirnames, filenames in os.walk(root_folder):
        for fname in filenames:
            if fname.lower().endswith(".gpkg"):
                gpkg_paths.append(os.path.join(dirpath, fname))
    return sorted(gpkg_paths)


def get_scene_info_from_path(filepath, root_folder):
    """Splits a per-date gpkg path into a 'scene' name (its phase subfolder,
    e.g. 'Morning' or 'Morning/merged') and a base name (the date, e.g. '24_8_2022')."""
    rel_path = os.path.relpath(filepath, root_folder)
    rel_dir = os.path.dirname(rel_path)
    base_name = os.path.splitext(os.path.basename(filepath))[0]
    scene_name = rel_dir if rel_dir else "root"
    return scene_name, base_name


def clean_value_field(gdf, field_name, treat_zero_as_nodata=False, zero_tol=0.0):
    gdf = gdf.copy()
    stats = {"removed_non_numeric_or_nan": 0, "removed_zero_as_nodata": 0}

    # Force float64 — esda's numba-jitted permutation code (crand) requires
    # float64 throughout and will raise a TypingError against float32 input
    # (e.g. columns written by Notebook 02 as float32).
    vals = pd.to_numeric(gdf[field_name], errors="coerce").astype(np.float64)
    stats["removed_non_numeric_or_nan"] = int((~np.isfinite(vals)).sum())

    gdf[field_name] = vals
    gdf = gdf[np.isfinite(gdf[field_name])].copy()

    if treat_zero_as_nodata:
        zero_mask = np.isclose(gdf[field_name].astype(float), 0.0, atol=zero_tol)
        stats["removed_zero_as_nodata"] = int(zero_mask.sum())
        gdf = gdf.loc[~zero_mask].copy()

    return gdf, stats


def build_distance_weights_binary(gdf, band):
    w = DistanceBand.from_dataframe(
        gdf,
        threshold=band,
        binary=True,
        silence_warnings=True
    )
    w.transform = "B"
    return w


def arcgis_gistar_bins(z_scores, p_values):
    bins = []
    for z, p in zip(z_scores, p_values):
        if np.isnan(z) or np.isnan(p) or p > 0.10:
            bins.append(0)
        elif z > 0:
            if p <= 0.01:
                bins.append(3)
            elif p <= 0.05:
                bins.append(2)
            else:
                bins.append(1)
        else:
            if p <= 0.01:
                bins.append(-3)
            elif p <= 0.05:
                bins.append(-2)
            else:
                bins.append(-1)
    return bins


def moran_cluster_labels(q, pvals, alpha=0.05):
    out = []
    for qi, pi in zip(q, pvals):
        if np.isnan(pi) or pi > alpha:
            out.append("Not Significant")
        elif qi == 1:
            out.append("High-High")
        elif qi == 2:
            out.append("Low-High")
        elif qi == 3:
            out.append("Low-Low")
        elif qi == 4:
            out.append("High-Low")
        else:
            out.append("Not Significant")
    return out


def make_gi_legend_handles():
    items = [
        (3, "Hot Spot with 99% confidence"),
        (2, "Hot Spot with 95% confidence"),
        (1, "Hot Spot with 90% confidence"),
        (0, "Not Significant"),
        (-1, "Cold Spot with 90% confidence"),
        (-2, "Cold Spot with 95% confidence"),
        (-3, "Cold Spot with 99% confidence"),
    ]
    return [Patch(facecolor=gi_color_map[v], edgecolor="black", linewidth=0.3, label=lab) for v, lab in items]


def make_moran_legend_handles():
    items = [
        ("High-High", "High-High"),
        ("Low-Low", "Low-Low"),
        ("High-Low", "High-Low"),
        ("Low-High", "Low-High"),
        ("Not Significant", "Not Significant"),
    ]
    return [Patch(facecolor=moran_color_map[v], edgecolor="black", linewidth=0.3, label=lab) for v, lab in items]


def decorate_gis_map(
    ax,
    legend_handles=None,
    legend_title=None,
    scale_bar=True,
    north_arrow=True,
    location_scalebar=(0.08, 0.01),
    location_north=(0.87, 0.87)
):
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()
    x_range = xmax - xmin
    y_range = ymax - ymin

    x_pad = x_range * 0.12
    y_pad = y_range * 0.12
    ax.set_xlim(xmin, xmax + x_pad)
    ax.set_ylim(ymin - y_pad, ymax + y_pad)
    xmin, xmax = ax.get_xlim()
    ymin, ymax = ax.get_ylim()

    if scale_bar:
        if x_range > 300000:
            length_km = 100
        elif x_range > 150000:
            length_km = 50
        elif x_range > 50000:
            length_km = 20
        else:
            length_km = 10

        length_m = length_km * 1000
        x_start = xmin + (xmax - xmin) * location_scalebar[0]
        y_start = ymin + (ymax - ymin) * location_scalebar[1]

        ax.plot(
            [x_start, x_start + length_m],
            [y_start, y_start],
            color="black", linewidth=4, solid_capstyle="butt"
        )
        ax.text(
            x_start + length_m / 2,
            y_start + y_range * 0.02,
            f"{length_km} km",
            ha="center", va="bottom", fontsize=10, fontweight="bold"
        )

    if north_arrow:
        ax.annotate(
            "N",
            xy=location_north,
            xytext=(location_north[0], location_north[1] - 0.15),
            xycoords="axes fraction",
            textcoords="axes fraction",
            ha="center", va="center", fontsize=14, fontweight="bold",
            arrowprops=dict(facecolor="black", edgecolor="black", width=1.5, headwidth=6, headlength=8)
        )

    ax.legend(
        handles=legend_handles,
        title=legend_title,
        loc="upper left",
        bbox_to_anchor=(1.02, 1),
        frameon=True,
        fontsize=8,
        title_fontsize=9,
        labelspacing=0.3,
        handletextpad=0.4,
        borderpad=0.4,
        borderaxespad=0.2
    )

    ax.axis("off")
    return ax

In [48]:
def run_hotspot_for_layer(
    gdf,
    scene_name,
    base_name,
    value_field,
    raw_path,
    png_path,
    layer_subfolder,          # "Per_Date" or "Phase_Average"
    grid_id_field=GRID_ID_FIELD
):
    """
    Runs Getis-Ord Gi* and Local Moran's I across all distance_bands for one
    layer (one date's zonal stats, or one phase's averaged column).

    Writes:
      - one raw .gpkg per distance band into raw_path/layer_subfolder
      - ONE combined PNG per layer into png_path/layer_subfolder, with
        rows = distance bands and columns = [Gi*, Moran], each with its own legend.

    Returns a list of summary dicts (one per distance band).
    """
    summaries = []

    if value_field not in gdf.columns:
        print(f"  WARNING: field '{value_field}' not found in {scene_name}/{base_name}. Skipping.")
        return summaries
    if gdf.empty:
        print(f"  WARNING: {scene_name}/{base_name} is empty. Skipping.")
        return summaries
    if gdf.crs is None:
        print(f"  WARNING: {scene_name}/{base_name} has no CRS. Skipping.")
        return summaries

    original_n = len(gdf)
    gdf, clean_stats = clean_value_field(
        gdf, value_field,
        treat_zero_as_nodata=treat_zero_as_nodata,
        zero_tol=zero_nodata_tolerance
    )
    removed_n = original_n - len(gdf)

    print(
        f"  Removed invalid features: non-numeric/NaN={clean_stats['removed_non_numeric_or_nan']}, "
        f"zero-as-NoData={clean_stats['removed_zero_as_nodata']}"
    )

    if len(gdf) < 10:
        print(f"  WARNING: too few valid features after cleaning ({len(gdf)} left). Skipping.")
        return summaries

    try:
        gdf = gdf.to_crs(epsg=target_epsg)
    except Exception as e:
        print(f"  ERROR reprojecting: {e}")
        return summaries

    run_timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    safe_scene = scene_name.replace(os.sep, "_").replace("/", "_")

    # Collect per-band results here; we plot everything together at the end
    band_results = []   # list of (band, gdf_band, gi_bin_label)

    for band in distance_bands:
        print(f"  Distance band: {band} m")

        try:
            w = build_distance_weights_binary(gdf, band)
        except Exception as e:
            print(f"    ERROR building weights: {e}")
            continue

        gdf_band = gdf.copy()

        # -------------------------
        # Getis-Ord Gi*
        # -------------------------
        try:
            gi = G_Local(gdf_band[value_field].values, w, transform="B", permutations=permutations, star=True)
        except TypeError:
            gi = G_Local(gdf_band[value_field].values, w, transform="B", permutations=permutations)

        gdf_band["Gi_Z"] = gi.Zs
        gdf_band["Gi_p_norm"] = gi.p_norm if hasattr(gi, "p_norm") else np.nan
        gdf_band["Gi_p_sim"] = gi.p_sim if hasattr(gi, "p_sim") else np.nan

        gdf_band["Gi_FDR_p_norm"] = np.nan
        valid_norm = np.isfinite(gdf_band["Gi_p_norm"])
        if valid_norm.any():
            gdf_band.loc[valid_norm, "Gi_FDR_p_norm"] = multipletests(
                gdf_band.loc[valid_norm, "Gi_p_norm"], alpha=0.05, method="fdr_bh"
            )[1]

        gdf_band["Gi_FDR_p_sim"] = np.nan
        valid_sim = np.isfinite(gdf_band["Gi_p_sim"])
        if valid_sim.any():
            gdf_band.loc[valid_sim, "Gi_FDR_p_sim"] = multipletests(
                gdf_band.loc[valid_sim, "Gi_p_sim"], alpha=0.05, method="fdr_bh"
            )[1]

        gdf_band["Gi_Bin_RawNorm"] = arcgis_gistar_bins(gdf_band["Gi_Z"], gdf_band["Gi_p_norm"])
        gdf_band["Gi_Bin_RawSim"] = arcgis_gistar_bins(gdf_band["Gi_Z"], gdf_band["Gi_p_sim"])
        gdf_band["Gi_Bin_FDRNorm"] = arcgis_gistar_bins(gdf_band["Gi_Z"], gdf_band["Gi_FDR_p_norm"])
        gdf_band["Gi_Bin_FDRSim"] = arcgis_gistar_bins(gdf_band["Gi_Z"], gdf_band["Gi_FDR_p_sim"])

        if gi_p_source_for_bins.lower() == "p_norm":
            gi_bin_label = "FDRNorm" if apply_fdr_to_gi_bins else "RawNorm"
        else:
            gi_bin_label = "FDRSim" if apply_fdr_to_gi_bins else "RawSim"
        gdf_band["Gi_Bin"] = gdf_band[f"Gi_Bin_{gi_bin_label}"]

        # -------------------------
        # Local Moran's I
        # -------------------------
        moran = Moran_Local(gdf_band[value_field].values, w, transformation="B", permutations=permutations)
        gdf_band["Local_I"] = moran.Is
        gdf_band["Local_p"] = moran.p_sim
        gdf_band["Local_q"] = moran.q

        if apply_fdr_to_moran:
            gdf_band["Local_FDR"] = np.nan
            valid_local = np.isfinite(gdf_band["Local_p"])
            if valid_local.any():
                gdf_band.loc[valid_local, "Local_FDR"] = multipletests(
                    gdf_band.loc[valid_local, "Local_p"], alpha=0.05, method="fdr_bh"
                )[1]
            moran_sig_field = "Local_FDR"
        else:
            gdf_band["Local_FDR"] = gdf_band["Local_p"]
            moran_sig_field = "Local_p"

        gdf_band["Moran_Clus"] = moran_cluster_labels(gdf_band["Local_q"], gdf_band[moran_sig_field], alpha=0.05)

        # Neighbor diagnostics
        neighbor_counts = np.array([len(v) for v in w.neighbors.values()]) if len(w.neighbors) else np.array([])
        min_neighbors = int(neighbor_counts.min()) if neighbor_counts.size else 0
        mean_neighbors = float(neighbor_counts.mean()) if neighbor_counts.size else 0.0
        max_neighbors = int(neighbor_counts.max()) if neighbor_counts.size else 0

        # Export raw results (GeoPackage) — still one file per band
        out_stub = f"{safe_scene}_{base_name}_{band}m"
        raw_out = os.path.join(raw_path, layer_subfolder, f"{out_stub}_raw.gpkg")
        gdf_band.to_file(raw_out, driver="GPKG")

        total_features = len(gdf_band)
        summary = {
            "Layer_Type": layer_subfolder,
            "Scene": scene_name,
            "File": base_name,
            "Value_Field": value_field,
            "Distance_m": band,
            "Run_Timestamp": run_timestamp,
            "Original_Features": original_n,
            "Removed_NonNumeric_or_NaN": clean_stats["removed_non_numeric_or_nan"],
            "Removed_ZeroAsNoData": clean_stats["removed_zero_as_nodata"],
            "Removed_Total_Invalid": removed_n,
            "Analyzed_Features": total_features,
            "Weights_Transform": "BINARY",
            "Gi_Bin_Source_Used": gi_bin_label,
            "Min_Neighbors": min_neighbors,
            "Mean_Neighbors": round(mean_neighbors, 2),
            "Max_Neighbors": max_neighbors,
            "Mean_GiZScore": float(np.nanmean(gdf_band["Gi_Z"])),
            "Median_GiZScore": float(np.nanmedian(gdf_band["Gi_Z"])),
            "StdDev_GiZScore": float(np.nanstd(gdf_band["Gi_Z"], ddof=1)),
            "Gi_Signif_RawNorm_0.05": int((gdf_band["Gi_p_norm"] <= 0.05).sum()),
            "Gi_Signif_RawSim_0.05": int((gdf_band["Gi_p_sim"] <= 0.05).sum()),
            "Gi_Signif_FDRNorm_0.05": int((gdf_band["Gi_FDR_p_norm"] <= 0.05).sum()),
            "Gi_Signif_FDRSim_0.05": int((gdf_band["Gi_FDR_p_sim"] <= 0.05).sum()),
            "GiBin_Hot99": int((gdf_band["Gi_Bin"] == 3).sum()),
            "GiBin_Hot95": int((gdf_band["Gi_Bin"] == 2).sum()),
            "GiBin_Hot90": int((gdf_band["Gi_Bin"] == 1).sum()),
            "GiBin_NotSig": int((gdf_band["Gi_Bin"] == 0).sum()),
            "GiBin_Cold90": int((gdf_band["Gi_Bin"] == -1).sum()),
            "GiBin_Cold95": int((gdf_band["Gi_Bin"] == -2).sum()),
            "GiBin_Cold99": int((gdf_band["Gi_Bin"] == -3).sum()),
            "Moran_HH": int((gdf_band["Moran_Clus"] == "High-High").sum()),
            "Moran_LL": int((gdf_band["Moran_Clus"] == "Low-Low").sum()),
            "Moran_HL": int((gdf_band["Moran_Clus"] == "High-Low").sum()),
            "Moran_LH": int((gdf_band["Moran_Clus"] == "Low-High").sum()),
            "Moran_NotSig": int((gdf_band["Moran_Clus"] == "Not Significant").sum()),
        }
        summaries.append(summary)
        band_results.append((band, gdf_band, gi_bin_label))

        print(f"    Stats computed for {band} m (raw gpkg saved).")

    # -------------------------
    # ONE combined PNG per layer: rows = distance bands, columns = [Gi*, Moran]
    # -------------------------
    if band_results:
        n_rows = len(band_results)
        fig, axes = plt.subplots(n_rows, 2, figsize=(20, 10 * n_rows))
        if n_rows == 1:
            axes = axes.reshape(1, 2)

        for row_idx, (band, gdf_band, gi_bin_label) in enumerate(band_results):
            ax_gi = axes[row_idx, 0]
            ax_moran = axes[row_idx, 1]

            # --- Gi* panel ---
            for val in [3, 2, 1, 0, -1, -2, -3]:
                subset = gdf_band[gdf_band["Gi_Bin"] == val]
                if not subset.empty:
                    subset.plot(ax=ax_gi, color=gi_color_map[val], edgecolor="black", linewidth=0.05)
            ax_gi.set_title(f"Getis-Ord Gi* ({gi_bin_label}) — {band} m", fontsize=12)
            decorate_gis_map(ax_gi, legend_handles=make_gi_legend_handles(), legend_title="Gi* Bins")

            # --- Moran panel ---
            for cat in ["High-High", "Low-Low", "High-Low", "Low-High", "Not Significant"]:
                subset = gdf_band[gdf_band["Moran_Clus"] == cat]
                if not subset.empty:
                    subset.plot(ax=ax_moran, color=moran_color_map[cat], edgecolor="black", linewidth=0.05)
            ax_moran.set_title(f"Anselin Local Moran's I — {band} m", fontsize=12)
            decorate_gis_map(ax_moran, legend_handles=make_moran_legend_handles(), legend_title="Cluster / Outlier")

        fig.suptitle(f"{scene_name} | {base_name}", fontsize=16, y=1.005)
        combined_out = os.path.join(png_path, layer_subfolder, f"{safe_scene}_{base_name}_combined.png")
        plt.savefig(combined_out, dpi=300, bbox_inches="tight", pad_inches=0.4)
        plt.close()
        print(f"    Combined map saved -> {combined_out}")

    return summaries

In [49]:
base_path, raw_path, summary_path, png_path = create_results_structure(INPUT_FOLDER)
print(f"Output folders ready: {base_path}")

Output folders ready: /Users/ks/Desktop/Wu/Testing/05_Hotspot_Analysis_Results


In [50]:
# =========================
# PART A: PER-DATE ZONAL STATS
# =========================

gpkg_files = find_all_gpkgs(ZONAL_STATS_DIR)
print(f"Found {len(gpkg_files)} per-date gpkg file(s) in {ZONAL_STATS_DIR}.")

per_date_summaries = []

for filepath in gpkg_files:
    scene_name, base_name = get_scene_info_from_path(filepath, ZONAL_STATS_DIR)
    print(f"\nProcessing per-date layer: {scene_name}/{base_name}")

    try:
        gdf = gpd.read_file(filepath)
    except Exception as e:
        print(f"  ERROR reading gpkg: {e}")
        continue

    scene_rows = run_hotspot_for_layer(
        gdf, scene_name, base_name,
        value_field=PER_DATE_VALUE_FIELD,
        raw_path=raw_path, png_path=png_path,
        layer_subfolder="Per_Date"
    )

    per_date_summaries.extend(scene_rows)

    if save_individual_scene_csv and scene_rows:
        safe_scene = scene_name.replace(os.sep, "_").replace("/", "_")
        pd.DataFrame(scene_rows).to_csv(
            os.path.join(summary_path, f"{safe_scene}_{base_name}_summary.csv"),
            index=False
        )

print(f"\nPart A complete: {len(per_date_summaries)} band-level result rows.")

Found 27 per-date gpkg file(s) in /Users/ks/Desktop/Wu/Testing/02_Ecostress_Zonal_Stats.

Processing per-date layer: Evening/12_7_2024
  Removed invalid features: non-numeric/NaN=4255, zero-as-NoData=0

Processing per-date layer: Evening/15_7_2024
  Removed invalid features: non-numeric/NaN=2694, zero-as-NoData=0
  Distance band: 1000 m


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/libpysal/weights/util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 17 disconnected components.
  w = W(neighbors, weights, ids, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/esda/getisord.py:450: RuntimeWarning: divide by zero encountered in divide
  self.z_sim = (self.Gs - self.EG_sim) / self.seG_sim
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/esda/moran.py:1351: RuntimeWarning: invalid value encountered in divide
  self.z_sim = (self.Is - self.EI_sim) / self.seI_sim


    Stats computed for 1000 m (raw gpkg saved).
  Distance band: 1500 m
    Stats computed for 1500 m (raw gpkg saved).
  Distance band: 2000 m
    Stats computed for 2000 m (raw gpkg saved).
    Combined map saved -> /Users/ks/Desktop/Wu/Testing/05_Hotspot_Analysis_Results/PNG_Maps/Per_Date/Evening_15_7_2024_combined.png

Processing per-date layer: Evening/16_7_2024
  Removed invalid features: non-numeric/NaN=0, zero-as-NoData=0
  Distance band: 1000 m


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/libpysal/weights/util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 23 disconnected components.
  w = W(neighbors, weights, ids, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/esda/getisord.py:450: RuntimeWarning: divide by zero encountered in divide
  self.z_sim = (self.Gs - self.EG_sim) / self.seG_sim
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/esda/moran.py:1351: RuntimeWarning: invalid value encountered in divide
  self.z_sim = (self.Is - self.EI_sim) / self.seI_sim


    Stats computed for 1000 m (raw gpkg saved).
  Distance band: 1500 m
    Stats computed for 1500 m (raw gpkg saved).
  Distance band: 2000 m
    Stats computed for 2000 m (raw gpkg saved).
    Combined map saved -> /Users/ks/Desktop/Wu/Testing/05_Hotspot_Analysis_Results/PNG_Maps/Per_Date/Evening_16_7_2024_combined.png

Processing per-date layer: Evening/20_7_2024
  Removed invalid features: non-numeric/NaN=33, zero-as-NoData=0
  Distance band: 1000 m


...

    Stats computed for 1500 m (raw gpkg saved).
  Distance band: 2000 m


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/libpysal/weights/util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 2 disconnected components.
  w = W(neighbors, weights, ids, **kwargs)


    Stats computed for 2000 m (raw gpkg saved).
    Combined map saved -> /Users/ks/Desktop/Wu/Testing/05_Hotspot_Analysis_Results/PNG_Maps/Per_Date/Night_merged_30_6_2024_combined.png

Processing per-date layer: Night/merged/4_7_2024
  Removed invalid features: non-numeric/NaN=0, zero-as-NoData=0
  Distance band: 1000 m


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/libpysal/weights/util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 23 disconnected components.
  w = W(neighbors, weights, ids, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/esda/getisord.py:450: RuntimeWarning: divide by zero encountered in divide
  self.z_sim = (self.Gs - self.EG_sim) / self.seG_sim
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/esda/moran.py:1351: RuntimeWarning: invalid value encountered in divide
  self.z_sim = (self.Is - self.EI_sim) / self.seI_sim


    Stats computed for 1000 m (raw gpkg saved).
  Distance band: 1500 m
    Stats computed for 1500 m (raw gpkg saved).
  Distance band: 2000 m
    Stats computed for 2000 m (raw gpkg saved).
    Combined map saved -> /Users/ks/Desktop/Wu/Testing/05_Hotspot_Analysis_Results/PNG_Maps/Per_Date/Night_merged_4_7_2024_combined.png

Part A complete: 48 band-level result rows.


In [51]:
# =========================
# PART B: PHASE-AVERAGED LAYERS (Night / Morning / Afternoon / Evening)
# =========================

phase_average_summaries = []

if not os.path.isfile(FINAL_AVERAGES_GPKG):
    print(f"WARNING: {FINAL_AVERAGES_GPKG} not found. Skipping Part B.")
else:
    final_gdf = gpd.read_file(FINAL_AVERAGES_GPKG)
    print(f"Loaded {len(final_gdf)} grid cells from {FINAL_AVERAGES_GPKG}")

    for phase in PHASE_CATEGORIES:
        if phase not in final_gdf.columns:
            print(f"\n  '{phase}' column not found in Final_Phase_Averages.gpkg. Skipping.")
            continue

        print(f"\nProcessing phase-average layer: {phase}")

        phase_gdf = final_gdf[[GRID_ID_FIELD, phase, "geometry"]].copy()

        scene_rows = run_hotspot_for_layer(
            phase_gdf, scene_name="Final_Phase_Average", base_name=phase,
            value_field=phase,
            raw_path=raw_path, png_path=png_path,
            layer_subfolder="Phase_Average"
        )

        phase_average_summaries.extend(scene_rows)

        if save_individual_scene_csv and scene_rows:
            pd.DataFrame(scene_rows).to_csv(
                os.path.join(summary_path, f"Final_Phase_Average_{phase}_summary.csv"),
                index=False
            )

print(f"\nPart B complete: {len(phase_average_summaries)} band-level result rows.")

Loaded 4255 grid cells from /Users/ks/Desktop/Wu/Testing/04_Final_Phase_Averages/Final_Phase_Averages.gpkg

Processing phase-average layer: Night
  Removed invalid features: non-numeric/NaN=0, zero-as-NoData=0
  Distance band: 1000 m


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/libpysal/weights/util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 23 disconnected components.
  w = W(neighbors, weights, ids, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/esda/getisord.py:450: RuntimeWarning: divide by zero encountered in divide
  self.z_sim = (self.Gs - self.EG_sim) / self.seG_sim
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/esda/moran.py:1351: RuntimeWarning: invalid value encountered in divide
  self.z_sim = (self.Is - self.EI_sim) / self.seI_sim


    Stats computed for 1000 m (raw gpkg saved).
  Distance band: 1500 m
    Stats computed for 1500 m (raw gpkg saved).
  Distance band: 2000 m
    Stats computed for 2000 m (raw gpkg saved).
    Combined map saved -> /Users/ks/Desktop/Wu/Testing/05_Hotspot_Analysis_Results/PNG_Maps/Phase_Average/Final_Phase_Average_Night_combined.png

Processing phase-average layer: Morning
  Removed invalid features: non-numeric/NaN=0, zero-as-NoData=0
  Distance band: 1000 m


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/libpysal/weights/util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 23 disconnected components.
  w = W(neighbors, weights, ids, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/esda/getisord.py:450: RuntimeWarning: divide by zero encountered in divide
  self.z_sim = (self.Gs - self.EG_sim) / self.seG_sim
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/esda/moran.py:1351: RuntimeWarning: invalid value encountered in divide
  self.z_sim = (self.Is - self.EI_sim) / self.seI_sim


    Stats computed for 1000 m (raw gpkg saved).
  Distance band: 1500 m
    Stats computed for 1500 m (raw gpkg saved).
  Distance band: 2000 m
    Stats computed for 2000 m (raw gpkg saved).
    Combined map saved -> /Users/ks/Desktop/Wu/Testing/05_Hotspot_Analysis_Results/PNG_Maps/Phase_Average/Final_Phase_Average_Morning_combined.png

  'Afternoon' column not found in Final_Phase_Averages.gpkg. Skipping.

Processing phase-average layer: Evening
  Removed invalid features: non-numeric/NaN=0, zero-as-NoData=0
  Distance band: 1000 m


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/libpysal/weights/util.py:826: UserWarning: The weights matrix is not fully connected: 
 There are 23 disconnected components.
  w = W(neighbors, weights, ids, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/esda/getisord.py:450: RuntimeWarning: divide by zero encountered in divide
  self.z_sim = (self.Gs - self.EG_sim) / self.seG_sim
/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/esda/moran.py:1351: RuntimeWarning: invalid value encountered in divide
  self.z_sim = (self.Is - self.EI_sim) / self.seI_sim


    Stats computed for 1000 m (raw gpkg saved).
  Distance band: 1500 m
    Stats computed for 1500 m (raw gpkg saved).
  Distance band: 2000 m
    Stats computed for 2000 m (raw gpkg saved).
    Combined map saved -> /Users/ks/Desktop/Wu/Testing/05_Hotspot_Analysis_Results/PNG_Maps/Phase_Average/Final_Phase_Average_Evening_combined.png

Part B complete: 9 band-level result rows.


In [52]:
# =========================
# COMBINE + SAVE MASTER SUMMARY
# =========================

all_summaries = per_date_summaries + phase_average_summaries
summary_df = pd.DataFrame(all_summaries)

run_id = datetime.now().strftime("%Y%m%d_%H%M%S")
summary_csv = os.path.join(summary_path, f"Hotspot_Summary_{run_id}.csv")
summary_df.to_csv(summary_csv, index=False)

print("\nAnalysis complete.")
print(f"Combined summary saved to:\n{summary_csv}")
summary_df


Analysis complete.
Combined summary saved to:
/Users/ks/Desktop/Wu/Testing/05_Hotspot_Analysis_Results/Summary_Tables/Hotspot_Summary_20260706_153424.csv


,Layer_Type,Scene,File,Value_Field,Distance_m,Run_Timestamp,Original_Features,Removed_NonNumeric_or_NaN,Removed_ZeroAsNoData,Removed_Total_Invalid,...,GiBin_Hot90,GiBin_NotSig,GiBin_Cold90,GiBin_Cold95,GiBin_Cold99,Moran_HH,Moran_LL,Moran_HL,Moran_LH,Moran_NotSig
0,Per_Date,Evening,15_7_2024,lst_mean,1000,2026-07-06 15:31:27,4255,2694,0,2694,...,75,927,50,202,0,293,218,0,0,1050
1,Per_Date,Evening,15_7_2024,lst_mean,1500,2026-07-06 15:31:27,4255,2694,0,2694,...,51,556,119,128,245,464,332,3,0,762
2,Per_Date,Evening,15_7_2024,lst_mean,2000,2026-07-06 15:31:27,4255,2694,0,2694,...,43,434,93,143,303,534,412,4,1,610
3,Per_Date,Evening,16_7_2024,lst_mean,1000,2026-07-06 15:31:35,4255,0,0,0,...,296,2115,184,368,461,785,781,0,0,2689
4,Per_Date,Evening,16_7_2024,lst_mean,1500,2026-07-06 15:31:35,4255,0,0,0,...,185,1305,165,307,938,1394,1219,0,0,1642
5,Per_Date,Evening,16_7_2024,lst_mean,2000,2026-07-06 15:31:35,4255,0,0,0,...,118,1075,105,330,1082,1547,1419,0,1,1288
6,Per_Date,Evening,20_7_2024,lst_mean,1000,2026-07-06 15:31:45,4255,33,0,33,...,251,2884,197,398,0,551,351,1,0,3319
7,Per_Date,Evening,20_7_2024,lst_mean,1500,2026-07-06 15:31:45,4255,33,0,33,...,223,1854,110,242,535,1132,739,11,1,2339
8,Per_Date,Evening,20_7_2024,lst_mean,2000,2026-07-06 15:31:45,4255,33,0,33,...,173,1675,111,249,605,1445,843,31,9,1894
9,Per_Date,Evening/merged,12_7_2024,lst_mean,1000,2026-07-06 15:31:56,4255,0,0,0,...,144,2633,190,420,207,688,545,1,0,3021
